<a href="https://colab.research.google.com/github/Aswin312006/ratingpredict.ipynb/blob/main/ratingpredict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout,
    Concatenate, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head(3))

Saving amazon.csv to amazon.csv
Dataset loaded successfully!
Dataset shape: (1465, 16)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...


In [3]:
print("Original dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Convert the rating to a numeric value
df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.strip(),
    errors="coerce"
)

# Clean numeric fields that are stored as text
for col in ["discounted_price", "actual_price"]:
    df[col] = pd.to_numeric(
        df[col].astype(str)
        .str.replace(r"[₹,$,\s]", "", regex=True),
        errors="coerce"
    )

df["discount_percentage"] = pd.to_numeric(
    df["discount_percentage"].astype(str)
    .str.replace("%", "", regex=False),
    errors="coerce"
)

df["rating_count"] = pd.to_numeric(
    df["rating_count"].astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Remove rows with no valid target rating
df = df.dropna(subset=["rating"]).copy()

# Ratings outside the expected range are not valid targets
df = df[df["rating"].between(1, 5)].copy()

# Fill missing product features
numeric_cols = [
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating_count"
]

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing text fields
text_cols = [
    "product_name",
    "category",
    "about_product",
    "review_title",
    "review_content"
]

for col in text_cols:
    df[col] = df[col].fillna("")

print("\nCleaned dataset shape:", df.shape)
print("\nMissing values in important columns:")
print(df[numeric_cols + ["rating"]].isnull().sum())
print("\nRating range:", df["rating"].min(), "to", df["rating"].max())

Original dataset shape: (1465, 16)

Missing values:
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

Cleaned dataset shape: (1464, 16)

Missing values in important columns:
discounted_price       0
actual_price           0
discount_percentage    0
rating_count           0
rating                 0
dtype: int64

Rating range: 2.0 to 5.0


In [4]:
# Extract the first category level
df["main_category"] = (
    df["category"]
    .str.split("|")
    .str[0]
    .replace("", "Unknown")
)

# Combine review text with product information
df["combined_text"] = (
    df["product_name"].astype(str) + " " +
    df["about_product"].astype(str) + " " +
    df["review_title"].astype(str) + " " +
    df["review_content"].astype(str)
).str.lower().str.strip()

# Remove rows with no meaningful text
df = df[
    df["combined_text"].str.len() > 0
].copy()

print("Records available:", len(df))
print("\nExample input text:")
print(df["combined_text"].iloc[0][:500])

print("\nTarget rating:")
print(df["rating"].iloc[0])

Records available: 1464

Example input text:
wayona nylon braided usb to lightning fast charging and data sync cable compatible for iphone 13, 12,11, x, 8, 7, 6, 5, ipad air, pro, mini (3 ft pack of 1, grey) high compatibility : compatible with iphone 12, 11, x/xsmax/xr ,iphone 8/8 plus,iphone 7/7 plus,iphone 6s/6s plus,iphone 6/6 plus,iphone 5/5s/5c/se,ipad pro,ipad air 1/2,ipad mini 1/2/3,ipod nano7,ipod touch and more apple devices.|fast charge&data sync : it can charge and sync simultaneously at a rapid speed, compatible with any charg

Target rating:
4.2


In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

Training records: 936
Validation records: 235
Testing records: 293


In [6]:
MAX_WORDS = 10000
MAX_LEN = 200

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Fit vocabulary on training text only
tokenizer.fit_on_texts(train_df["combined_text"])

# Convert words into integer sequences
X_train_text = tokenizer.texts_to_sequences(
    train_df["combined_text"]
)
X_val_text = tokenizer.texts_to_sequences(
    val_df["combined_text"]
)
X_test_text = tokenizer.texts_to_sequences(
    test_df["combined_text"]
)

# Pad/truncate all sequences to a fixed length
X_train_text = pad_sequences(
    X_train_text, maxlen=MAX_LEN,
    padding="post", truncating="post"
)
X_val_text = pad_sequences(
    X_val_text, maxlen=MAX_LEN,
    padding="post", truncating="post"
)
X_test_text = pad_sequences(
    X_test_text, maxlen=MAX_LEN,
    padding="post", truncating="post"
)

vocab_size = min(
    MAX_WORDS,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size:", vocab_size)
print("Maximum sequence length:", MAX_LEN)
print("Training text shape:", X_train_text.shape)
print("Validation text shape:", X_val_text.shape)
print("Testing text shape:", X_test_text.shape)

# Display one example
print("\nOriginal review:")
print(train_df["combined_text"].iloc[0][:300])

print("\nEncoded sequence:")
print(X_train_text[0][:30])

Vocabulary size: 10000
Maximum sequence length: 200
Training text shape: (936, 200)
Validation text shape: (235, 200)
Testing text shape: (293, 200)

Original review:
7seven compatible lg tv remote suitable for lg non magic smart tv remote control (mouse & voice non-support) mr20ga prime video and netflix hotkeys for lg tv series all models of remote control compatible|comapatible models: it is applicable for your lg smart tv universal remote an-mr600g an-mr650 a

Encoded sequence:
[5138  113  757   49  105  463    6  757  480 1786  100   49  105  180
  142  261  480  135 9733  758  219    3  854 2264    6  757   49  826
   43  969]


In [7]:
# Numeric product information
scaler = StandardScaler()

X_train_num = scaler.fit_transform(
    train_df[numeric_cols].astype(float)
)
X_val_num = scaler.transform(
    val_df[numeric_cols].astype(float)
)
X_test_num = scaler.transform(
    test_df[numeric_cols].astype(float)
)

# Encode product categories
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    # Compatibility with older scikit-learn versions
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

X_train_cat = encoder.fit_transform(
    train_df[["main_category"]]
).astype("float32")

X_val_cat = encoder.transform(
    val_df[["main_category"]]
).astype("float32")

X_test_cat = encoder.transform(
    test_df[["main_category"]]
).astype("float32")

# Target ratings
y_train = train_df["rating"].to_numpy(dtype="float32")
y_val = val_df["rating"].to_numpy(dtype="float32")
y_test = test_df["rating"].to_numpy(dtype="float32")

print("Training numeric features:", X_train_num.shape)
print("Training category features:", X_train_cat.shape)
print("Training ratings:", y_train.shape)

Training numeric features: (936, 4)
Training category features: (936, 6)
Training ratings: (936,)


In [8]:
# Input 1: Encoded review text
text_input = Input(
    shape=(MAX_LEN,),
    name="review_input"
)

text_features = Embedding(
    input_dim=vocab_size,
    output_dim=64,
    mask_zero=True,
    name="word_embedding"
)(text_input)

text_features = LSTM(
    64,
    name="lstm_layer"
)(text_features)

text_features = Dropout(0.35)(text_features)

# Input 2: Numerical product features
numeric_input = Input(
    shape=(len(numeric_cols),),
    name="numeric_input"
)

numeric_features = Dense(
    16,
    activation="relu"
)(numeric_input)

# Input 3: Product category
category_input = Input(
    shape=(X_train_cat.shape[1],),
    name="category_input"
)

category_features = Dense(
    16,
    activation="relu"
)(category_input)

# Combine all feature branches
combined = Concatenate()([
    text_features,
    numeric_features,
    category_features
])

combined = Dense(64, activation="relu")(combined)
combined = BatchNormalization()(combined)
combined = Dropout(0.30)(combined)

combined = Dense(32, activation="relu")(combined)

# One continuous rating output
output = Dense(
    1,
    activation="linear",
    name="predicted_rating"
)(combined)

model = Model(
    inputs=[text_input, numeric_input, category_input],
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ review_input        │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 200, 64)   │    640,000 │ review_input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ review_input[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer (LSTM)   │ (None, 64)        │     33,024 │ word_embedding[0… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_input      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lstm_layer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         80 │ numeric_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        112 │ category_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense[0][0],      │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      6,208 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ predicted_rating    │ (None, 1)         │         33 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 681,793 (2.60 MB)

 Trainable params: 681,665 (2.60 MB)

 Non-trainable params: 128 (512.00 B)

In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=0.00001,
    verbose=1
)

history = model.fit(
    x={
        "review_input": X_train_text,
        "numeric_input": X_train_num,
        "category_input": X_train_cat
    },
    y=y_train,
    validation_data=(
        {
            "review_input": X_val_text,
            "numeric_input": X_val_num,
            "category_input": X_val_cat
        },
        y_val
    ),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\nModel training completed!")
print("Epochs completed:", len(history.history["loss"]))

Epoch 1/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 13s 159ms/step - loss: 7.4305 - mae: 2.4960 - val_loss: 13.1998 - val_mae: 3.6203 - learning_rate: 0.0010
Epoch 2/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 143ms/step - loss: 1.5192 - mae: 1.0144 - val_loss: 10.9073 - val_mae: 3.2882 - learning_rate: 0.0010
Epoch 3/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 197ms/step - loss: 0.9232 - mae: 0.7554 - val_loss: 10.1414 - val_mae: 3.1701 - learning_rate: 0.0010
Epoch 4/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 167ms/step - loss: 0.7730 - mae: 0.6959 - val_loss: 9.2937 - val_mae: 3.0333 - learning_rate: 0.0010
Epoch 5/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 167ms/step - loss: 0.7362 - mae: 0.6837 - val_loss: 8.1134 - val_mae: 2.8311 - learning_rate: 0.0010
Epoch 6/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 143ms/step - loss: 0.5649 - mae: 0.5870 - val_loss: 7.2612 - val_mae: 2.6763 - learning_rate: 0.0010
Epoch 7/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 191ms/step - loss: 0.5747 - mae: 0.5928 - val_loss: 6.4900 - val_mae: 2.5271 - learning_rate: 0.0010
Ep

In [10]:
test_inputs = {
    "review_input": X_test_text,
    "numeric_input": X_test_num,
    "category_input": X_test_cat
}

# Predict ratings for the unseen test set
raw_predictions = model.predict(test_inputs, verbose=0).flatten()

# Restrict displayed predictions to the valid rating range
y_pred = np.clip(raw_predictions, 1, 5)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION RESULTS")
print("------------------------")
print(f"MAE        : {mae:.4f}")
print(f"RMSE       : {rmse:.4f}")
print(f"R2 SCORE   : {r2:.4f}")
print(f"TEST SAMPLES: {len(y_test)}")

MODEL EVALUATION RESULTS
------------------------
MAE        : 0.2936
RMSE       : 0.3564
R2 SCORE   : -0.6372
TEST SAMPLES: 293


In [11]:
print("PREDICTION VARIATION CHECK")
print("--------------------------")
print("Actual rating standard deviation:",
      round(np.std(y_test), 4))

print("Predicted rating standard deviation:",
      round(np.std(y_pred), 4))

print("\nFirst 15 predictions:")
display(pd.DataFrame({
    "Actual Rating": y_test[:15],
    "Predicted Rating": np.round(y_pred[:15], 2),
    "Absolute Error": np.round(
        np.abs(y_test[:15] - y_pred[:15]), 2
    )
}))

PREDICTION VARIATION CHECK
--------------------------
Actual rating standard deviation: 0.2786
Predicted rating standard deviation: 0.243

First 15 predictions:


,Actual Rating,Predicted Rating,Absolute Error
0,4.2,3.94,0.26
1,3.7,3.89,0.19
2,3.3,3.82,0.52
3,4.3,4.28,0.02
4,3.9,3.66,0.24
5,4.5,3.75,0.75
6,4.1,3.75,0.35
7,4.1,3.96,0.14
8,4.2,3.72,0.48
9,3.6,3.86,0.26


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Baseline: predict the average training rating for every test product
baseline_pred = np.full(len(y_test), np.mean(y_train))

# Compare baseline and LSTM predictions
comparison_results = []

for name, predictions in [
    ("Mean Baseline", baseline_pred),
    ("LSTM Model", y_pred)
]:
    comparison_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2 Score": r2_score(y_test, predictions)
    })

display(pd.DataFrame(comparison_results).round(4))

,Model,MAE,RMSE,R2 Score
0,Mean Baseline,0.2101,0.2787,-0.0013
1,LSTM Model,0.2936,0.3564,-0.6372


In [13]:

import numpy as np

actual = np.asarray(y_test)
predicted = np.asarray(predictions)

# Percentage of predictions within ±0.5 rating points
correct_predictions = np.abs(actual - predicted) <= 0.5
accuracy = np.mean(correct_predictions) * 100

print(f"Accuracy : {accuracy:.2f}%")
print(f"Total Loss (MAE) : {np.mean(np.abs(actual - predicted)):.4f}")


Accuracy : 85.32%
Total Loss (MAE) : 0.2936


In [15]:

# TEST THE LSTM MODEL WITH USER INPUT

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Get review details from the user
product_name = input("Enter product name: ")
review_title = input("Enter review title: ")
review_content = input("Enter your product review: ")
category = input("Enter product category: ")

# Combine text in the same format used during training
user_text = (
    product_name + " " +
    review_title + " " +
    review_content
).lower()

# Convert text into a padded sequence
sequence = tokenizer.texts_to_sequences([user_text])
text_input = pad_sequences(
    sequence,
    maxlen=200,
    padding="post",
    truncating="post"
)

# Prepare numerical features using training medians
numeric_values = []
for col in numeric_cols:
    if col == "discounted_price":
        value = float(input("Enter discounted price: "))
    elif col == "actual_price":
        value = float(input("Enter actual price: "))
    elif col == "discount_percentage":
        value = float(input("Enter discount percentage: "))
    elif col == "rating_count":
        value = float(input("Enter rating count: "))
    else:
        value = 0.0
    numeric_values.append(value)

# Transform numerical values using the existing scaler
numeric_input = scaler.transform(
    np.array(numeric_values).reshape(1, -1)
)

# Encode category using the existing encoder
category_input = encoder.transform([[category]])

# Predict the rating
predicted_rating = model.predict(
    {
        "review_input": text_input,
        "numeric_input": numeric_input,
        "category_input": category_input
    },
    verbose=0
)[0][0]

predicted_rating = np.clip(predicted_rating, 1, 5)

print("\n----- PRODUCT RATING PREDICTION -----")
print("Product Name:", product_name)
print("Predicted Rating:", round(float(predicted_rating), 2), "/ 5")


Enter product name: boAt
Enter review title: Excellent sound quality
Enter your product review: The sound is clear and the bass is good. Battery life is impressive and the headphones are comfortable for daily use.
Enter product category: Electronics
Enter discounted price: 1499
Enter actual price: 3990
Enter discount percentage: 62
Enter rating count: 12500

----- PRODUCT RATING PREDICTION -----
Product Name: boAt
Predicted Rating: 3.85 / 5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [17]:
import pandas as pd
import numpy as np

# Transform numerical inputs without feature-name warnings
numeric_input = scaler.transform(
    pd.DataFrame([numeric_values], columns=numeric_cols)
)

# Transform category input without feature-name warnings
category_column = (
    encoder.feature_names_in_[0]
    if hasattr(encoder, "feature_names_in_")
    else "main_category"
)

category_input = encoder.transform(
    pd.DataFrame([[category]], columns=[category_column])
)

# Convert sparse matrix to dense array if needed
if hasattr(category_input, "toarray"):
    category_input = category_input.toarray()

print("Numerical and category inputs processed successfully!")

Numerical and category inputs processed successfully!
